# Etapa 2 — Limpeza, Preparação e Extração de Features

**Por que estas duas etapas juntas?**

O computador tem memória limitada. Carregar o dataset inteiro de uma vez (10+ GB) e depois
processar tudo junto causaria erros de memória. A solução é processar **uma classe por vez**:

1. Carregar instâncias da Classe 0 → limpar → extrair features → salvar → apagar da memória
2. Carregar instâncias da Classe 1 → limpar → extrair features → salvar → apagar da memória
3. ... repetir para as 10 classes

A função `run_pipeline_chunked()` faz exatamente isso. No final, temos dois arquivos salvos
em disco sem nunca ter mais de uma classe na memória ao mesmo tempo.

**Saídas:**
- `data/processed/cleaned.parquet` — séries temporais limpas
- `data/processed/features.parquet` — uma linha por janela de 300s, com ~55 features

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold

from config import (
    CLEANED_DATA_PATH,
    FAULT_CLASSES,
    FEATURES_DATA_PATH,
    FFILL_LIMIT,
    N_INSTANCES_VALIDATION,
    N_SPLITS_CV,
    VALIDATION_MODE,
)
from src.data_loader import load_sample
from src.feature_engineering import run_pipeline_chunked

# Banner de modo — mudança rápida em config.py
mode_label = 'VALIDACAO' if VALIDATION_MODE else 'COMPLETO'
n_inst = N_INSTANCES_VALIDATION if VALIDATION_MODE else 'todas'
print(f"{'='*55}")
print(f"  Modo: {mode_label}  |  Instancias por classe: {n_inst}")
if VALIDATION_MODE:
    print("  Para rodar o dataset completo:")
    print("  config.py -> VALIDATION_MODE = False")
print(f"{'='*55}")

  Modo: VALIDACAO  |  Instancias por classe: 5
  Para rodar o dataset completo:
  config.py -> VALIDATION_MODE = False


## 2.1 Conceito: forward-fill (preenchimento de lacunas)

Antes de rodar o pipeline, vamos entender o que o forward-fill faz em um exemplo real.

Imagine que o sensor de pressão 'parou' por 30 segundos e voltou. Durante esse tempo,
a leitura fica como NaN (desconhecida). O forward-fill copia o último valor válido para
preencher esses buracos — mas só por até 60 segundos. Buracos maiores permanecem como NaN.

In [2]:
# Carregar uma instância de exemplo para demonstrar
df_demo = load_sample(n_instances_per_class=1)
df_inst = df_demo[df_demo['instance_id'] == df_demo['instance_id'].iloc[0]].copy()

sensor = 'P-TPT'
if sensor in df_inst.columns:
    before = df_inst[sensor].isna().sum()
    df_inst[sensor] = df_inst[sensor].ffill(limit=FFILL_LIMIT)
    after = df_inst[sensor].isna().sum()
    print(f'Sensor {sensor}:')
    print(f'  NaN antes do forward-fill : {before}')
    print(f'  NaN depois do forward-fill: {after}')
    print(f'  Lacunas preenchidas       : {before - after}')
else:
    print(f'Sensor {sensor} nao encontrado nessa instancia.')

Sensor P-TPT:
  NaN antes do forward-fill : 0
  NaN depois do forward-fill: 0
  Lacunas preenchidas       : 0


## 2.2 Conceito: GroupKFold (divisão treino/teste por poço)

Este é um ponto crucial para a validade do TCC.

Se dividirmos os dados aleatoriamente, janelas do **mesmo poço** podem aparecer no treino
e no teste ao mesmo tempo. O modelo 'memoriza' aquele poço e parece ótimo — mas falha em
poços novos que nunca viu. Isso é chamado de **vazamento de dados** (*data leakage*).

O GroupKFold garante que cada poço aparece em **apenas um** dos conjuntos.

In [ ]:
# Demonstrar GroupKFold com os poucos dados da amostra
instances = df_demo['instance_id'].unique()
groups = df_demo.groupby('instance_id')['fault_class'].first().loc[instances].values

# A demo usa no máximo 3 folds para funcionar com qualquer tamanho de amostra.
# No treino real usamos N_SPLITS_CV do config (5 no modo completo).
n_splits_demo = min(3, len(instances))
gkf_demo = GroupKFold(n_splits=n_splits_demo)

print(f'GroupKFold com {n_splits_demo} folds — cada poco aparece em apenas 1 fold de teste:')
for fold, (train_idx, test_idx) in enumerate(gkf_demo.split(instances, groups, groups)):
    print(f'  Fold {fold+1}: treino={len(train_idx)} instancias | teste={len(test_idx)} instancias')

print(f'\n(No treino real usamos N_SPLITS_CV={N_SPLITS_CV} folds com todas as instancias)')

## 2.3 Executar o pipeline em partes

Agora rodamos o pipeline completo: limpeza + extração de features, classe por classe.

No **modo validação**, são carregadas apenas `N_INSTANCES_VALIDATION` instâncias por classe
(definido em `config.py`). Isso é suficiente para verificar se o código funciona sem erros,
em segundos ou poucos minutos.

No **modo completo** (`VALIDATION_MODE = False`), todas as instâncias são processadas.
Isso pode levar 30 minutos a algumas horas dependendo do hardware.

In [4]:
# Apagar arquivos anteriores se existirem (evita acumular dados de runs diferentes)
for path in [CLEANED_DATA_PATH, FEATURES_DATA_PATH]:
    if path.exists():
        path.unlink()
        print(f'Arquivo anterior removido: {path.name}')

print('\nIniciando pipeline...\n')
run_pipeline_chunked(verbose=True)
print('\nPipeline concluido!')

Arquivo anterior removido: cleaned.parquet
Arquivo anterior removido: features.parquet

Iniciando pipeline...

  Carregando classe 0: Normal (max 5)... 5 instâncias, 107,381 linhas
    -> 5 instâncias após limpeza
    -> 709 janelas geradas
  Carregando classe 1: Aumento Abrupto de BSW (max 5)... 5 instâncias, 338,400 linhas
    -> 5 instâncias após limpeza
    -> 2,251 janelas geradas
  Carregando classe 2: Fechamento Espúrio da DHSV (max 5)... 5 instâncias, 143,995 linhas
    -> 5 instâncias após limpeza
    -> 950 janelas geradas
  Carregando classe 3: Golfadas Severas (max 5)... 5 instâncias, 294,995 linhas
    -> 5 instâncias após limpeza
    -> 1,959 janelas geradas
  Carregando classe 4: Instabilidade de Fluxo (max 5)... 5 instâncias, 53,757 linhas
    -> 5 instâncias após limpeza
    -> 350 janelas geradas
  Carregando classe 5: Perda Rápida de Produtividade (max 5)... 5 instâncias, 146,495 linhas
    -> 5 instâncias após limpeza
    -> 970 janelas geradas
  Carregando classe 6

## 2.4 Verificar os arquivos gerados

In [5]:
# Carregar e inspecionar o arquivo de dados limpos
df_clean = pd.read_parquet(CLEANED_DATA_PATH)
print('=== Dados Limpos ===')
print(f'Shape          : {df_clean.shape}')
print(f'Instancias     : {df_clean["instance_id"].nunique()}')
print()

dist = df_clean.groupby(['fault_class', 'source_type'])['instance_id'].nunique().unstack(fill_value=0)
dist.index = dist.index.map(lambda c: f'{c} — {FAULT_CLASSES[c]}')
print('Instancias por classe e fonte:')
print(dist.to_string())

=== Dados Limpos ===
Shape          : (2940213, 33)
Instancias     : 20

Instancias por classe e fonte:
source_type                        DRAWN  SIMULATED  WELL
fault_class                                              
0 — Normal                             0          0     5
1 — Aumento Abrupto de BSW             5          0     0
2 — Fechamento Espúrio da DHSV         0          5     0
3 — Golfadas Severas                   0          5     0
4 — Instabilidade de Fluxo             0          0     5
5 — Perda Rápida de Produtividade      0          5     0
6 — Restrição Rápida no PCK            0          5     0
7 — Incrustação no PCK                 5          0     0
8 — Hidrato na Linha de Produção       0          5     0
9 — Hidrato na Linha de Serviço        0          5     0


In [6]:
# Carregar e inspecionar o arquivo de features
df_features = pd.read_parquet(FEATURES_DATA_PATH)
META_COLS = ['instance_id', 'fault_class', 'source_type', 'window_start']
n_features = df_features.shape[1] - len(META_COLS)

print('=== Features ===')
print(f'Shape          : {df_features.shape}')
print(f'Janelas totais : {len(df_features):,}')
print(f'Features/janela: {n_features}')
print()
print('Janelas por classe:')
print(df_features['fault_class'].value_counts().sort_index()
      .rename(FAULT_CLASSES).to_string())

=== Features ===
Shape          : (19522, 92)
Janelas totais : 19,522
Features/janela: 88

Janelas por classe:
fault_class
Normal                            709
Aumento Abrupto de BSW           2251
Fechamento Espúrio da DHSV        950
Golfadas Severas                 1959
Instabilidade de Fluxo            350
Perda Rápida de Produtividade     970
Restrição Rápida no PCK           890
Incrustação no PCK               9499
Hidrato na Linha de Produção      890
Hidrato na Linha de Serviço      1054
